# Setup

In [1]:
import os

import gymnasium as gym
import pandas as pd
from skopt.space import Real
import time

from popy.simulation_tools import *
from popy.io_tools import load_behavior
from popy.behavior_data_tools import *
from popy.simulation_helpers import fit_simulate, fit_agent, fit_agent_graddesc
from popy.config import PROJECT_PATH_LOCAL

/home/uzsombi/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-frrtiskr because there was an issue with the default path (/home/uzsombi/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Init

In [2]:
# Load data
from skopt.space import Integer

monkey = 'ka'
env = gym.make("zsombi/monkey-bandit-task-v0", n_arms=3, max_episode_steps=100_000)
behav_monkey = load_behavior(monkey)
behav_monkey = drop_time_fields(behav_monkey)
behav_monkey = add_switch_info(behav_monkey)
behav_monkey = convert_column_format(behav_monkey, original='behavior')
behav_monkey = behav_monkey.dropna()

print(f"Loaded {len(behav_monkey)} trials from monkey {monkey}")

"""# Define model and parameters
fit_params = {
    "alpha": Real(0.01, .7, name="alpha"),
    "beta": Real(0.5, 15.0, name="beta"),
    "V0": Real(0.05, 0.4, name="V0"),
}"""

# Foraging model
'''agent_class = ForagingAgent
fixed_params = {"reset_on_switch": True}
free_params = ["alpha", "beta", "V0"]
param_space = [fit_params[p] for p in free_params]'''

from skopt.space import Real

fit_params = {
    "alpha_pos": Real(0.01, 0.7, name="alpha_pos"),
    "alpha_neg": Real(0.01, 0.7, name="alpha_neg"),
    "beta": Real(2.0, 15.0, name="beta"),  # base beta
    "stickiness": Real(0.0, 1.0, name="stickiness"),
}

agent_class = AdvancedForagingAgent
fixed_params = {
    "memory_length": 3,
    "adaptive_beta": True,
}
free_params = ["alpha_pos", "alpha_neg", "beta", "stickiness"]
param_space = [fit_params[p] for p in free_params]

Loaded 23860 trials from monkey ka


# Simulate models

## gp_minimize

In [3]:
# GP Minimize approach (fit_agent)
print("\n=== GP Minimize (Bayesian Optimization) ===")

gp_params = {
    "n_calls": 250,
    "n_initial_points": 100,
    "n_jobs": -1,
    "verbose": True,
}

start_time = time.time()
result_gp = fit_agent(
    agent_class=agent_class,
    param_space=param_space,
    env=env,
    behav_data=behav_monkey,
    fixed_params=fixed_params,
    fit_on="ll",
    n_calls=gp_params["n_calls"],
    n_initial_points=gp_params["n_initial_points"],
    n_jobs=gp_params["n_jobs"],
    verbose=gp_params["verbose"],
)
gp_time = time.time() - start_time

print(f"Best parameters: {result_gp['best_params']}")
print(f"Best LL: {result_gp['best_ll']:.4f}")
print(f"BIC: {result_gp['bic']:.4f}")
print(f"LPT: {result_gp['lpt']:.4f}")
print(f"Time: {gp_time:.2f}s")


=== GP Minimize (Bayesian Optimization) ===
Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 2.9073
Function value obtained: 8243.0642
Current minimum: 8243.0642
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 2.6178
Function value obtained: 11357.1965
Current minimum: 8243.0642
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 2.5918
Function value obtained: 9432.3308
Current minimum: 8243.0642
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 2.6288
Function value obtained: 8512.6741
Current minimum: 8243.0642
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 2.5099
Function value obtain

KeyboardInterrupt: 

## sklearn optimize.minimize

In [ ]:
# Gradient Descent approach (fit_agent_graddesc)
print("\n=== Gradient Descent (with restarts) ===")

# Extract bounds and names for graddesc function
bounds = [(p.low, p.high) for p in param_space]
param_names_list = [p.name for p in param_space]

gd_params = {
    "n_restarts": 1,
    "patience": 5,
    "options": {},
    "method": "Nelder-Mead",
}

start_time = time.time()
result_gd = fit_agent_graddesc(
    agent_class=agent_class,
    bounds=bounds,
    param_names=param_names_list,
    env=env,
    behav_data=behav_monkey,
    fixed_params=fixed_params,
    fit_on="ll",
    n_restarts=gd_params["n_restarts"],
    patience=gd_params["patience"],
    options=gd_params["options"],
    method=gd_params["method"],
    verbose=True,
)
gd_time = time.time() - start_time

print(f"Best parameters: {result_gd['best_params']}")
print(f"Best LL: {result_gd['best_ll']:.4f}")
print(f"BIC: {result_gd['bic']:.4f}")
print(f"LPT: {result_gd['lpt']:.4f}")
print(f"Time: {gd_time:.2f}s")

print(f"\nSpeedup: {gp_time/gd_time:.2f}x")
print(f"performance improvement: {(result_gd['best_ll'] / result_gp['best_ll']):.4f} LL points")